In [0]:
# Databricks notebook source

# =============================================================================
# Projeto........: Parts Handbook
# Notebook.......: 00_nb_sales_order_ingestion
# Camada.........: raw
# Objetivo.......: Realizar a carga histórica da Sales Order a partir dos
#                  arquivos Parquet exportados do SAP, aplicando a
#                  padronização dos dados e inicializando a tabela
#                  raw_sales_order.
#
# Desenvolvedor..: André Causs
# Criado em......: 25/07/2026
#
# Premissas
# -----------------------------------------------------------------------------
# • Arquivos históricos em formato Parquet.
# • Carga inicial (bootstrap) da camada RAW.
# • Normalização dos nomes de colunas.
# • Inclusão de metadados de origem e auditoria.
# • Persistência em Delta Lake utilizando overwrite.
# • Execução única, sem Structured Streaming.
#
# Histórico de Alterações
# -----------------------------------------------------------------------------
# Data       Autor          Versão  Descrição
# ---------- -------------- ------- --------------------------------------------
# 25/07/2026 André Causs    1.0.0   Criação da estrutura inicial.
# =============================================================================

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

import re
import unicodedata
import uuid

CATALOG = "parts_hdbk_sandbox"
SCHEMA = "dt_sales_orders"
RAW_TABLE = f"{CATALOG}.{SCHEMA}.raw_sales_order"

HISTORICAL_PATH = (
    "/Volumes/parts_hdbk_sandbox/dt_sales_orders/sap_sales_order/historical"
)

BUSINESS_KEY = ["numero_ov", "item"]
LOAD_MODE = "INITIAL_LOAD"

RAW_BUSINESS_COLUMNS = [
    "numero_ov", "data", "tipo_ov", "motivo_ov", "bloqueio_rem",
    "motivo_recusa", "org_vendas", "canal_dist", "setor_ativ", "centro",
    "emissor_da_ordem", "numero_pedido", "autor", "item", "material",
    "categoria_do_item", "item_superior", "quantidade", "um",
]
SOURCE_METADATA_COLUMNS = [
    "_source_file_name", "_source_file_path", "_source_file_modification_time",
]
AUDIT_COLUMNS = [
    "_ingested_at", "_last_updated_at", "_ingested_by", "_load_type", "_load_id",
]
RAW_COLUMNS = RAW_BUSINESS_COLUMNS + SOURCE_METADATA_COLUMNS + AUDIT_COLUMNS
LOAD_ID = str(uuid.uuid4())

# Deixe desabilitado na carga produtiva: cada validação abaixo lê a tabela novamente.
VALIDATE_AFTER_WRITE = False

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
CURRENT_USER = spark.sql("SELECT current_user() AS user").first()["user"]


In [0]:
def normalize_column_name(column_name: str) -> str:
    normalized = unicodedata.normalize("NFKD", column_name)
    normalized = normalized.encode("ascii", "ignore").decode("ascii")
    normalized = re.sub(r"[^a-z0-9]", "_", normalized.lower())
    return re.sub(r"_+", "_", normalized).strip("_")


def normalize_dataframe_columns(df: DataFrame) -> DataFrame:
    return df.toDF(*[normalize_column_name(column) for column in df.columns])


def read_historical_sales_order() -> DataFrame:
    # recursiveFileLookup é necessário porque o histórico pode conter subdiretórios.
    return (
        spark.read
        .format("parquet")
        .option("recursiveFileLookup", "true")
        .load(HISTORICAL_PATH)
    )


def transform_sales_order(df: DataFrame) -> DataFrame:
    df = normalize_dataframe_columns(df)

    required_columns = set(RAW_BUSINESS_COLUMNS)
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes: {sorted(missing_columns)}")

    standardized_df = df.withColumns({
        "quantidade": F.coalesce(
            F.col("quantidade").cast("decimal(18,3)"),
            F.lit(0).cast("decimal(18,3)"),
        ),
        "_source_file_name": F.col("_metadata.file_name"),
        "_source_file_path": F.col("_metadata.file_path"),
        "_source_file_modification_time": F.col("_metadata.file_modification_time"),
        "_ingested_at": F.from_utc_timestamp(
            F.current_timestamp(), "America/Sao_Paulo"
        ),
        "_last_updated_at": F.from_utc_timestamp(
            F.current_timestamp(), "America/Sao_Paulo"
        ),
        "_ingested_by": F.lit(CURRENT_USER),
        "_load_type": F.lit(LOAD_MODE),
        "_load_id": F.lit(LOAD_ID),
    })

    return standardized_df.select(*RAW_COLUMNS)


In [0]:
def write_historical_sales_order(df: DataFrame) -> None:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(RAW_TABLE)
    )


def validate_historical_load() -> None:
    # Execute somente quando for necessário auditar a carga.
    duplicate_count = (
        spark.table(RAW_TABLE)
        .groupBy(*BUSINESS_KEY)
        .count()
        .where(F.col("count") > 1)
        .count()
    )
    total_rows = spark.table(RAW_TABLE).count()
    print(f"Linhas carregadas: {total_rows}")
    print(f"Chaves duplicadas: {duplicate_count}")


In [0]:
source_df = read_historical_sales_order()
historical_df = transform_sales_order(source_df)

# Esta é a única ação sobre os arquivos de origem.
write_historical_sales_order(historical_df)

last_commit = (
    DeltaTable.forName(spark, RAW_TABLE)
    .history(1)
    .select("version", "timestamp", "operation", "operationMetrics")
    .first()
)
print(f"Carga histórica concluída. Versão Delta: {last_commit['version']}")
print(f"Métricas da escrita: {last_commit['operationMetrics']}")

if VALIDATE_AFTER_WRITE:
    validate_historical_load()


In [0]:
spark.sql(f"""
    COMMENT ON TABLE {RAW_TABLE} IS
    'Camada Raw da Sales Order SAP. Dados normalizados e auditados sem regras de negócio.'
""")
